In [0]:
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/07_dim_reason
# Purpose  : Unified Dimension for encounter/medication/procedure reasons (SNOMED-CT).
#
# Simplified Design Decision: Implemented as a Materialized View (Batch) instead of
# Streaming + apply_changes. This enables combining the Unknown Member (-1) with actual
# codes in a single query without mixing Streaming/Batch sources. Due to its small size,
# full recomputation adds virtually zero overhead (aligned with dim_date design).
#
# Deterministic Surrogate Keys are guaranteed across runs using xxhash64(reason_code).

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.dim_reason",
    comment="Reason dimension - unified SNOMED-CT reasons from encounters/medications/procedures + Unknown Member row (-1)",
)
def dim_reason():
    encounters = spark.read.table("covid19_lakehouse.silver.encounters").select(
        "reason_code", "reason_description"
    )
    medications = spark.read.table("covid19_lakehouse.silver.medications").select(
        "reason_code", "reason_description"
    )
    procedures = spark.read.table("covid19_lakehouse.silver.procedures").select(
        "reason_code", "reason_description"
    )

    real_reasons = (
        encounters.unionByName(medications)
        .unionByName(procedures)
        .filter(F.col("reason_code").isNotNull())
        .dropDuplicates(["reason_code"])
        .select(
            F.xxhash64("reason_code").alias("reason_key"),
            "reason_code",
            "reason_description",
        )
    )

    unknown_member = spark.createDataFrame(
        [(-1, "N/A", "No reason recorded")],
        schema="reason_key long, reason_code string, reason_description string",
    )

    return real_reasons.unionByName(unknown_member)